<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
<br>汉化的库: <a href="https://github.com/GoatCsu/CN-LLMs-from-scratch.git">https://github.com/GoatCsu/CN-LLMs-from-scratch.git</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第三章: Attention

本章所需要的包

In [2]:
from importlib.metadata import version
print("torch version:", version("torch"))
#导入并确认库

torch version: 2.11.0+cu128


- LLM的核心:Attention
- 译者:可以直接看论文!
- [Attention is all you need](https://arxiv.org/abs/1706.03762)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/01.webp?123" width="500px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/02.webp" width="600px">

## 3.1 长序列的建模

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/03.webp" width="400px">

- 在Transformer模型出现之前，机器翻译任务主要依赖于编码器(encoder)-解码器(decoder)架构的循环神经网络（RNNs）。
- 在这种架构中，编码器逐词处理源语言序列，并通过隐藏状态（神经网络中的中间层）生成输入序列的表示：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/04.webp" width="500px">

## 3.2 注意力机制高效捕获数据关系

- 本节不涉及代码。
- 借助注意力机制，文本生成解码器能够选择性地关注所有输入token，从而在生成特定输出token时，动态分配不同输入token的重要性系数

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/05.webp" width="500px">

- Transformer中的自注意力机制是一种关键技术，它通过让序列中的每个位置与其他所有位置交互并计算相关性，从而增强输入表示的上下文信息。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/06.webp" width="300px">

## 3.3 自注意力关注的不同部分

### 3.3.1 无可变参数的自注意力模型

- 本节介绍了一种高度简化的自注意力变体，不包含任何可训练的权重。
- 该变体仅用于说明目的，并非Transformer中实际使用的注意力机制。
- 下一节（3.3.2节）将扩展此简易模型，实现真正的自注意力机制。
- 假设给定一个输入序列 $x^{(1)}$ 到 $x^{(T)}$：
  - 输入是一个文本（例如，一句已被处理为token嵌入的句子，如“Your journey starts with one step”），具体处理方法在第2章中已有描述。
  - 例如，$x^{(1)}$ 是表示单词“Your”的d维向量，以此类推。

- **目标：** 为输入序列中的每个元素 $x^{(i)}$（从 $x^{(1)}$ 到 $x^{(T)}$）计算上下文向量 $z^{(i)}$（$z$ 和 $x$ 的维度相同）。
    - 上下文向量 $z^{(i)}$ 是对输入 $x^{(1)}$ 到 $x^{(T)}$ 的加权求和。
    - 上下文向量是针对特定输入的“上下文”相关表示。
      - 以第二个输入 $x^{(2)}$ 为例，说明具体计算过程。
      - 第二个上下文向量 $z^{(2)}$ 是对所有输入 $x^{(1)}$ 到 $x^{(T)}$ 的加权求和，权重由相对于 $x^{(2)}$ 的注意力权重决定。
      - 注意力权重决定了每个输入元素对 $z^{(2)}$ 的贡献程度。
      - 简而言之，$z^{(2)}$ 是 $x^{(2)}$ 的增强版本，融合了与当前任务相关的所有其他输入元素的信息。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/07.webp" width="400px">

- （请注意，此图中的数字已截断至小数点后一位，以减少视觉干扰；其他图表中的数值也可能经过类似处理。）

- 按照惯例，未归一化的注意力值称为 **“注意力得分”**，而归一化后总和为1的注意力得分称为 **“注意力权重”**。

- 下方代码逐步演示了上图的操作过程
<br>

- **步骤 1：** 计算未归一化的注意力得分 $\omega$
- 假设使用第二个输入token作为查询，即 $q^{(2)} = x^{(2)}$，通过点积计算未归一化的注意力得分：
    - $\omega_{21} = x^{(1)} \cdot q^{(2)\top}$
    - $\omega_{22} = x^{(2)} \cdot q^{(2)\top}$
    - $\omega_{23} = x^{(3)} \cdot q^{(2)\top}$
    - ...
    - $\omega_{2T} = x^{(T)} \cdot q^{(2)\top}$
- 其中，$\omega$ 是希腊字母“欧米伽”，表示未归一化的注意力得分。
    - 在 $\omega_{21}$ 中，下标“21”表示以第2个元素为查询，与第1个元素计算得分。

- 假设我们有以下输入句子，该句子已根据第3章的描述嵌入到3维向量中（此处我们使用了一个非常小的嵌入维度进行说明，以便内容可以显示在页面上）： 

In [3]:
import torch
# 对一句话中的每个词定义了一个三维向量
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

- （在本书中，我们遵循机器学习和深度学习的常见惯例：训练样本以行表示，特征值以列表示；对于上述张量，每一行表示一个词，每一列表示一个嵌入维度。）

- 本节的主要目标是演示如何以第二个输入序列 $x^{(2)}$ 作为查询，计算其上下文向量 $z^{(2)}$。

- 图中展示了该过程的第一步，即通过点积操作计算 $x^{(2)}$ 与所有其他输入元素之间的注意力得分 $\omega$。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/08.webp" width="400px">

- 我们以输入序列中的第2个元素 $x^{(2)}$ 为例，计算其上下文向量 $z^{(2)}$；稍后会将此方法推广至计算所有上下文向量。
- 第一步是通过计算查询 $x^{(2)}$ 与所有输入token的点积，得到未归一化的注意力得分：

In [4]:
query = inputs[1]  # 取出第二个token作为查询向量query
print(query)
print(inputs.shape[0])  # shape[0]:张量的行, shape[0]:张量的列
attn_scores_2 = torch.empty(inputs.shape[0])  # 初始化一个空张量，用来存储注意力得分
for i, x_i in enumerate(inputs):              # 遍历 inputs 中的每一行
    attn_scores_2[i] = torch.dot(x_i, query)  # 计算两个一维向量的点积
    # attention分数，即两个向量的相似度，从公式上看也就是点积，越大说明越相似

print(attn_scores_2) # query和所有6个token的点积结果 (query和自己（第2个token，索引1）的点积是 1.4950，是所有得分里最高的)

tensor([0.5500, 0.8700, 0.6600])
6
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


- 补充说明：点积本质上是逐元素相乘并将所得积相加的一种简写表示：

In [5]:
res = 0.
for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]  # 逐元素计算
print(res)  # 手动计算结果
print(torch.dot(inputs[0], query))  # 点积计算结果
#相当于解释了一遍点成的内部原理

tensor(0.9544)
tensor(0.9544)


- **步骤 2：** 将未归一化的注意力得分（“欧米伽”，$\omega$）归一化，使其总和为1。
- 以下是一种简单的方法，用于将未归一化的注意力得分归一化：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/09.webp" width="500px">

In [6]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()  # 直接除以总和做归一化
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


- 然而，在实践中，使用softmax函数进行归一化更为常见，因为它能够更好地处理极端值，并且在训练过程中具有更理想的梯度特性，因此推荐使用。
- 下面是一个简单的softmax函数实现，用于缩放并对向量元素进行归一化，使它们的和为1：

In [7]:
def softmax_naive(x):  # 手动实现的 Softmax
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())
#用SoftMax做归一化, 处理好极端值
#有合理的梯度数据表现力

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


- 上述简单实现可能会因输入值过大或过小而遭遇数值不稳定问题，导致溢出或下溢。
- 因此，在实践中，建议使用PyTorch内置的softmax函数，因为它经过高度优化，性能更佳：

In [8]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("注意力权重:", attn_weights_2)
print("Sum:", attn_weights_2.sum())
#内置了数值稳定处理（减去最大值再指数，避免溢出）

注意力权重: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


- **步骤 3**：通过将嵌入的输入标记 $x^{(i)}$ 与注意力权重相乘，并对结果向量求和，计算上下文向量 $z^{(2)}$：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/10.webp" width="500px">

In [9]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)  # 初始化一个和 query 形状相同的全零向量，用来存储加权求和的结果  (形状是 [d]，和单个 token 的嵌入维度一致)
for i,x_i in enumerate(inputs):
    print('权重值:',attn_weights_2[i], '词元向量:',x_i, '乘积:',attn_weights_2[i] * x_i)
    context_vec_2 += attn_weights_2[i] * x_i
print('query2的上下文向量:',context_vec_2)  # q2的上下文向量: 所有加权向量的和

权重值: tensor(0.1385) 词元向量: tensor([0.4300, 0.1500, 0.8900]) 乘积: tensor([0.0596, 0.0208, 0.1233])
权重值: tensor(0.2379) 词元向量: tensor([0.5500, 0.8700, 0.6600]) 乘积: tensor([0.1308, 0.2070, 0.1570])
权重值: tensor(0.2333) 词元向量: tensor([0.5700, 0.8500, 0.6400]) 乘积: tensor([0.1330, 0.1983, 0.1493])
权重值: tensor(0.1240) 词元向量: tensor([0.2200, 0.5800, 0.3300]) 乘积: tensor([0.0273, 0.0719, 0.0409])
权重值: tensor(0.1082) 词元向量: tensor([0.7700, 0.2500, 0.1000]) 乘积: tensor([0.0833, 0.0270, 0.0108])
权重值: tensor(0.1581) 词元向量: tensor([0.0500, 0.8000, 0.5500]) 乘积: tensor([0.0079, 0.1265, 0.0870])
query2的上下文向量: tensor([0.4419, 0.6515, 0.5683])


### 3.3.2 计算所有token的attention score

#### 将其推广到所有输入序列标记：

- 上面，我们为输入2计算了注意力权重和上下文向量（如下图中高亮的行所示）。
- 接下来，我们将此计算推广到所有输入序列标记，计算对应的注意力权重和上下文向量。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/11.webp" width="400px">

- （请注意，图中的数字已四舍五入到小数点后两位；每一行的数值应相加为1.0或100%；其他图中的数字也进行了类似处理。）

- 在自注意力机制中，首先计算注意力得分，随后对这些得分进行归一化，得到总和为1的注意力权重。
- 接着，利用这些注意力权重对输入进行加权求和，生成上下文向量。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/12.webp" width="400px">

- 对所有成对元素应用之前的**步骤 1**，计算未归一化的注意力得分矩阵：

In [10]:
attn_scores = torch.empty(6, 6)#建立空表来储存相关联程度
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)#一点点计算相关性并输入表格
print(attn_scores)#事实上就是实现了两个单词之间的关联度列表输出

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


- 如果是矩阵相乘那么更有效率

In [11]:
attn_scores = inputs @ inputs.T  # 每一行和每一行做点积
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


- 与**第二步**相似, 我们对每一行都要归一化操作:

In [12]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


- 一个快速验证

In [13]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("第二行总和:", row_2_sum)
print("所有行总和:", attn_weights.sum(dim=-1)) # 验证一下大家加起来都是1

第二行总和: 1.0
所有行总和: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


- 用**step 3** 计算所有的向量:

In [14]:
all_context_vecs = attn_weights @ inputs  # 权重表格×输入向量, 一次性算出 6个单词 的上下文向量
print(all_context_vecs)  # 每行代表 每个词元的注意力权重 与 所有词向量 的加权和

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


-  作为合理性检查，之前计算的上下文向量 $z^{(2)} = [0.4419, 0.6515, 0.5683]$ 可以在上图的第二行找到：

In [15]:
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


## 3.4 可调整参数的自注意力机制

- 以下的概念框架展示了本节中开发的自注意力机制,以及这种机制是如何融入本书和本章的整体叙述与结构。

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/13.webp" width="400px">

### 3.4.1 手把手的计算attention的值

- 在本节中，我们实现了原始 Transformer 架构、GPT 模型以及大多数流行 LLM 中使用的自注意力机制。  
- 这种自注意力机制被称为“缩放点积注意力”（scaled dot-product attention）。  
- 整体思路与之前相似：  
  - 我们希望计算针对特定输入元素的上下文向量，即输入向量的加权和。  
  - 为此，我们需要生成注意力权重。  
- 如你所见，与之前介绍的基本注意力机制相比，只有一些细微差异：  
  - 最显著的区别是引入了在模型训练过程中更新的权重矩阵。  
  - 这些可训练的权重矩阵至关重要，它们使模型（尤其是注意力模块）能够学习生成“优质”的上下文向量。


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/14.webp" width="600px">

- 按照步骤实现自注意力机制，我们将首先介绍三个训练权重矩阵 $W_q$、$W_k$ 和 $W_v$。  
- 这三个矩阵用于通过矩阵乘法将嵌入的输入标记 $x^{(i)}$ 映射到查询向量、键向量和值向量：
- (译者: 分别是Query、Key、Value,专有名词)   

  - 查询向量：$q^{(i)} = W_q \,x^{(i)}$  
  - 键向量：$k^{(i)} = W_k \,x^{(i)}$  
  - 值向量：$v^{(i)} = W_v \,x^{(i)}$  


- 输入 $x$ 和查询向量 $q$ 的嵌入维度可以相同，也可以不同，具体取决于模型的设计和实现方式。
- 在 GPT 模型中，输入和输出维度通常是相同的，但为了便于示范并更好地理解计算过程，这里我们选择了不同的输入和输出维度：

In [16]:
x_2 = inputs[1]         # 取第二个词的向量，作为当前要处理的词
d_in = inputs.shape[1]  # 输入向量的维度，这里是3（每个词向量有3个数字）
d_out = 2               # 输出向量的维度，这里设为2（把3维向量映射成2维）

- 下面，我们初始化三个权重矩阵；请注意，为了简化输出并便于示范，我们将 `requires_grad=False`
- 但如果我们要在模型训练中使用这些权重矩阵，应将 `requires_grad=True`，以便在训练过程中更新这些矩阵。

In [17]:
torch.manual_seed(123)  # 固定随机种子确保可复现性
# 初始化三个权重矩阵[3,2]，这里先关闭梯度，只是演示
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

- 计算这三个向量值

In [18]:
query_2 = x_2 @ W_query  # 生成第2个词的Query向量
key_2   = x_2 @ W_key    # 生成第2个词的Key向量
value_2 = x_2 @ W_value  # 生成第2个词的Value向量
print(query_2)  # (3,) @ (3,2) = (2,)

tensor([0.4306, 1.4551])


- 我们可以清晰地看到,embedding被降维了:

In [19]:
keys = inputs @ W_key      # 生成了所有词的 Key 向量，存在 keys 里
values = inputs @ W_value  # 生成了所有词的 Value 向量，存在 values 里
print("k向量形状:", keys.shape)
print("v向量形状:", values.shape)
#中途检验下

k向量形状: torch.Size([6, 2])
v向量形状: torch.Size([6, 2])


- 在下一步 **步骤 2** 中，我们通过计算查询向量和每个键向量之间的点积来计算未归一化的注意力得分：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/15.webp" width="600px">

In [20]:
keys_2 = keys[1]  # 取出第2个词自己的 Key 向量
attn_score_22 = query_2.dot(keys_2)  # 第 2 个词的 Query 向量，和第 2 个词自己的 Key 向量的相似度
print(attn_score_22)

tensor(1.8524)


- 因为我们有六个输入,所以我们有六个attention score

In [21]:
attn_scores_2 = query_2 @ keys.T  # 一次性计算和所有 Key 的点积
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/16.webp" width="600px">

- 接下来，在 **步骤 3** 中，我们使用之前提到的 softmax 函数计算注意力权重（归一化后的注意力得分，总和为 1）。
- 与之前的不同之处在于，我们现在通过将注意力得分除以嵌入维度的平方根 $\sqrt{d_k}$（即 `d_k**0.5`）来对注意力得分进行缩放：

In [22]:
d_k = keys.shape[1]  # 取出 Key 向量的维度，这里是2
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)  # 对注意力得分做缩放，也就是除以√d_k（这里就是除以√2 ≈ 1.414）
                                                                  # 对缩放后的得分做 Softmax，把它们转成和为 1 的注意力权重
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/17.webp" width="600px">

- 在**第四步**, 我们可以计算每一个token的向量了:

In [23]:
context_vec_2 = attn_weights_2 @ values  # 用权重乘以每个词的 Value 向量，再把结果加起来，得到第 2 个词的上下文向量
print(context_vec_2) # (1,6) @ (6,2) = (1,2)
'''
完整流程串起来
你现在已经走完了标准自注意力的完整流程：
1.生成 Q/K/V 向量 → query_2, keys, values
2.计算注意力得分 → attn_scores_2 = query_2 @ keys.T
3.缩放 + Softmax 归一化 → attn_weights_2 = softmax(attn_scores_2 / √d_k)
4.加权求和得到上下文向量 → context_vec_2 = attn_weights_2 @ values
这就是 Transformer 里缩放点积注意力的核心逻辑！
'''

tensor([0.3061, 0.8210])


'\n完整流程串起来\n你现在已经走完了标准自注意力的完整流程：\n1.生成 Q/K/V 向量 → query_2, keys, values\n2.计算注意力得分 → attn_scores_2 = query_2 @ keys.T\n3.缩放 + Softmax 归一化 → attn_weights_2 = softmax(attn_scores_2 / √d_k)\n4.加权求和得到上下文向量 → context_vec_2 = attn_weights_2 @ values\n这就是 Transformer 里缩放点积注意力的核心逻辑！\n'

### 3.4.2 自注意模块

- 下面是代码

In [24]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        # 定义QKV的随机权重矩阵 (nn.Parameter 会自动把这三个矩阵注册为可训练参数，PyTorch 会自动计算梯度)
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        # 用输入 x 乘以三个权重矩阵，得到所有词的 queries、keys、values 向量
        keys    = x @ self.W_key
        queries = x @ self.W_query
        values  = x @ self.W_value
        attn_scores = queries @ keys.T  # 一次性算出所有词两两之间的点积得分
        attn_weights = torch.softmax(   # 除以 √d_k 做缩放； 对每一行做归一化，得到注意力权重
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vec = attn_weights @ values  # 一次性算出所有词的上下文向量
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/18.webp" width="400px">

- 我们可以使用 PyTorch 的 `Linear` 层简化上述实现，禁用偏置项后，`Linear` 层相当于矩阵乘法。
- 使用 `nn.Linear` 替代手动使用 `nn.Parameter(torch.rand(...))` 的一个主要优势是，`nn.Linear` 具有推荐的权重初始化方案，这有助于模型训练更加稳定。

In [25]:
'''
为什么推荐用 nn.Linear？
·自动初始化：PyTorch 会用推荐的初始化方案（如 Kaiming/Xavier 初始化），比手动 torch.rand 更稳定，训练时梯度更不容易消失/爆炸。
·代码更简洁：不用自己管理 nn.Parameter，PyTorch 会自动注册参数，方便后续模型保存和加载。
·符合工程规范：这是工业界 Transformer 实现里的标准写法，后续扩展多头注意力也更方便。
'''
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        # 计算注意力得分与权重：queries @ keys.T → 缩放除以 √d_k → softmax 归一化，步骤和上一版完全相同
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        # 加权求和：attn_weights @ values 得到上下文向量，和上一版结果等价
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


- `SelfAttention_v1` 和 `SelfAttention_v2` 会给出不同的输出，因为它们使用了不同的初始权重矩阵。

## 3.5 对未出现的信息的隐藏

- 普通自注意力里，每个词可以和所有词计算注意力，包括它后面的词
- 但因果自注意力（比如 GPT 里用的），为了保证预测时不泄露未来信息，必须让每个词只能看到自己和它前面的词，看不到后面的词
- 实现方法就是：把注意力矩阵中 “当前词后面的位置” 全部 mask 掉（设为 0 或负无穷）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/19.webp" width="400px">

### 3.5.1 因果自注意力机制

- 在这一节中，我们将把之前的自注意力机制转换为因果自注意力机制。
- 因果自注意力确保模型在预测序列中某个位置的值时，仅依赖于前面已知位置的输出，而不依赖于后续位置。
- 换句话说，这确保了每个下一个词的预测仅依赖于前面的词。
- 为了实现这一点，对于每个给定的标记，我们会将“未知的信息”（即输入文本中当前token之后的token）掩蔽掉：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="600px">

- 为了说明和实现因果自注意力，让我们使用上一节中的注意力得分和权重：

In [26]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)  # (6,6) ，此时每个词的权重分布里，包含了对所有词(含后面未来词)的注意力

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


- 隐藏未知信息的attention score最简单的方法是通过 PyTorch 的 `tril` 函数进行掩蔽，其中主对角线以下的元素（包括对角线本身）设置为 1，主对角线以上的元素设置为 0：

In [27]:
context_length = attn_scores.shape[0]  # (6,)
mask_simple = torch.tril(torch.ones(context_length, context_length))  # 生成下三角掩码矩阵
print(mask_simple)  # 因果掩码：1 代表 “可以看到”，0 代表 “看不到”

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


- 然后，我们可以将注意力权重与这个mask相乘，以将对角线以上的注意力得分置为零：

In [28]:
masked_simple = attn_weights * mask_simple  # 逐元素相乘 (掩码为1的位置不变,掩码为0的位置变0)
print(masked_simple)
#简单的效果图

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


- 如果在 softmax 之后进行掩蔽，它会破坏 softmax 所创建的概率分布。
- softmax 确保所有输出值的总和为 1。
- 如果在 softmax 之后进行掩蔽，就需要重新归一化输出，确保其总和为 1，这会使过程更加复杂，并可能带来意想不到的效果。

- 我们可以用以下方式确保所有的数据都是归一化的  (重新归一化)

In [29]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)  # 掩码后重新softmax

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


- 尽管我们在技术上已经完成了因果注意力机制的编码，但让我们简要地探讨一种更高效的方法，以实现与上述相同的效果。
- 因此，在注意力得分进入 softmax 函数之前，我们可以将对角线以上的未归一化注意力得分用负无穷大进行掩蔽，而不是将其置零并重新归一化：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/21.webp" width="450px">

In [30]:
# triu:上三角 / tril:下三角
print(torch.triu(torch.ones(6,6)))
print(torch.triu(torch.ones(6,6), diagonal=1)) # diagonal=偏移量
# 核心思想: 在Softmax之前，把未来位置的注意力得分设为负无穷，而不是在Softmax之后乘 0
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)  # torch.triu 生成上三角矩阵，diagonal=1 表示主对角线以上的位置为 1，主对角线及以下为 0
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)  # masked_fill 把有掩码的位置替换为 -inf（负无穷）
print(masked)

tensor([[1., 1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.]])
tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


- 结果显然是归一化的

In [31]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)  # 输出的矩阵和方式1的结果完全一样

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### 3.5.2 使用Dropout防止过拟合

- 此外，我们还应用了丢弃（Dropout）来减少训练过程中的过拟合。
- dropout可以应用于多个位置：
  - 例如，在计算注意力权重之后；
  - 或在将注意力权重与值向量相乘之后。
- 在这里，我们选择在计算注意力权重之后应用丢弃掩码，因为这种做法更为常见。

- 另外，在此示例中，我们使用了50%的丢弃率，这意味着随机屏蔽掉一半的注意力权重。（在后续训练GPT模型时，我们会使用更低的丢弃率，例如0.1或0.2。）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/22.webp" width="400px">

- 如果我们应用0.5（50%）的丢弃率，未被抛弃的值将相应地被缩放一个因子，1/0.5 = 2。
- 这种缩放通过公式 1 / (1 - `dropout_rate`) 计算得出。

In [32]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)  # 丢弃率 50% 的 Dropout 层
example = torch.ones(6, 6)       # 创建一个 6x6 的全1矩阵
print(dropout(example))  # 输出需被放大相应的倍数,以维持恒定 (可调试看结果)

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [33]:
torch.manual_seed(123)
print(dropout(attn_weights))
# 注意：应用 Dropout 后，注意力权重矩阵的行和不再是 1 了，这是正常现象，因为 Dropout 是训练时的随机操作，不影响推理阶段

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


- 生成的输出可能会因操作系统的不同而有所不同；
- 你可以在 [PyTorch 问题追踪器](https://github.com/pytorch/pytorch/issues/121595) 上了解更多内容。

### 3.5.3 实现一个简洁的因果自注意力

- 现在，我们准备实现一个完整的自注意力机制，包含因果掩码和dropout。
- 另一项任务是实现代码以处理包含多个输入的批次，确保我们的 `CausalAttention` 类能够支持第二章中实现的数据加载器所生成的批量输出。
- 为了简化起见，我们通过复制输入文本示例来模拟批量输入：

In [34]:
# 之前的inputs是单个序列，形状为[6, 3]
batch = torch.stack((inputs, inputs), dim=0)  # 把两个相同的tensor按第0维堆叠，生成批量数据
print(batch.shape)  # 形状变为[2, 6, 3]，表示：2 个批次，每个批次有 6 个词，每个词的向量维度是 3

torch.Size([2, 6, 3])


- 缩放因子  \sqrt{d}  的引入解决了注意力机制中的数值不稳定问题。
- 它确保了即使嵌入维度  d  较大，点积得分也能被合理地控制在一个适当范围，方便 Softmax 生成平滑的注意力分布，且梯度不会过大或过小。

In [35]:
class CausalAttention(nn.Module):  # 完整的因果自注意力模块
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)  # 新增Dropout层，防止过拟合
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))  # (6,6)的因果掩码矩阵（上三角矩阵）
        '''
        预先生成因果掩码（上三角矩阵），并注册为buffer（起名mask，通过self.mask调用）
        torch.triu(..., diagonal=1) 生成一个上三角矩阵，主对角线以上的位置为 1，标记“未来位置”
        register_buffer: 注册不可训练的固定张量
                         会把这个掩码存为模型的一部分，不参与梯度更新，但会跟着模型设备移动（比如从 CPU 到 GPU），避免每次前向传播都重新生成掩码，提升效率
        '''
    def forward(self, x):  # x形状：(2,6,3)
        b, num_tokens, d_in = x.shape  # 获取输入的形状：b=批次大小，num_tokens=词数，d_in=维度
        queries = self.W_query(x)  # 形状：(2,6,2)
        keys    = self.W_key(x)
        values  = self.W_value(x)
        # 计算注意力得分  形状：(2,6,2) @ (2,2,6) = (2,6,6)  结果:每个批次中，每个词和所有词的注意力得分矩阵
        attn_scores = queries @ keys.transpose(1, 2)  # keys.transpose(1,2) 把形状从(2,6,2)转为(2,2,6)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # 把掩码为True的位置（未来位置）的得分设为负无穷，Softmax后会变为0
                                              #  ↑ mask[:6, :6] 动态裁剪掩码，掩码只需要取 前6行×前6列 就够了. 让模型支持任意长度句子，不会报错
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)  # 缩放+Softmax归一化
        attn_weights = self.dropout(attn_weights)  # 应用Dropout防止过拟合的
        # 加权求和，得到上下文向量 形状：[2, 6, 2]
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
context_length = batch.shape[1]  # 句子长度：6
ca = CausalAttention(d_in, d_out, context_length, 0.0)  # d_in=3, d_out=2, context_length=6, dropout=0.0（不丢弃）
context_vecs = ca(batch)  # 输入批量数据，得到上下文向量
print(context_vecs)
print("上下文向量形状:", context_vecs.shape)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
上下文向量形状: torch.Size([2, 6, 2])


- Dropout仅在训练时要使用,验证时不需要

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/23.webp" width="500px">

## 3.6 拓展单头至多方注意

### 3.6.1 堆叠多个单头注意力层

- 以下是之前实现的自注意力机制总结（为简化起见，未展示因果和dropout掩码）：

- 这种机制也称为单头注意力：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/24.webp" width="400px">

- 我们通过堆叠多个单头注意力模块来构建多头注意力模块：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/25.webp" width="400px">

- 多头注意力的核心思想是使用不同的学习到的线性投影，并行地多次运行注意力机制。这使得模型能够在不同位置同时关注来自不同表示子空间的信息。
- 多头注意力的本质是：
1. 用多个独立的自注意力头，并行学习不同的注意力模式（比如一个头关注语法，一个头关注语义等）
2. 把每个头的输出向量拼接起来，得到更丰富的上下文表示

在 Python 中，super().__init__() 是一种调用父类（基类）构造函数的方法，常用于类继承的场景中。它确保子类能够正确初始化父类的属性和方法。

In [36]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)])
        '''
        创建 num_heads 个独立的 CausalAttention 头
        nn.ModuleList: 用来存放多个 nn.Module（这里就是多个注意力头），PyTorch 会自动管理它们的参数
        每个 CausalAttention 头都是独立的，有自己的 Q/K/V 权重，会学习不同的注意力模式
        '''
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)  # torch.cat(..., dim=-1)：把所有头的输出，在 最后一个维度（词向量维度）上拼接

torch.manual_seed(123)
context_length = batch.shape[1] # 词元数 6
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)

print(context_vecs)
print("词向量形状:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
词向量形状: torch.Size([2, 6, 4])


- 在上述实现中，嵌入维度为4，因为我们将 `d_out=2` 作为键、查询和值向量以及上下文向量的嵌入维度。由于使用了2个注意力头，输出嵌入维度为 2 * 2 = 4。

### 3.6.2 利用权重拆分实现多头注意力

- 尽管上述实现是一种直观且功能完整的多头注意力机制（通过封装之前的单头注意力 `CausalAttention` 实现），我们仍可以编写一个独立的 `MultiHeadAttention` 类来实现相同的功能。

- 在这个独立的 `MultiHeadAttention` 类中，我们不会将单个注意力头进行拼接。
- 相反，我们会创建独立的 W_query、W_key 和 W_value 权重矩阵，并将它们拆分为每个注意力头的单独矩阵：

In [37]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"  # 断言：输出维度必须能被头数整除，否则无法平均拆分   用法：assert条件, "条件不成立时的报错信息"
        self.d_out = d_out          # 最终输出的总维度
        self.num_heads = num_heads  # 注意力头的数量
        self.head_dim = d_out // num_heads  # 每个头的维度
        # 只需要3个线性层，一次性计算所有头的QKV   形状：(3,2)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # 输出投影层：把所有头的结果融合，学习不同头信息的最优组合方式
        self.dropout = nn.Dropout(dropout)       # Dropout层
        self.register_buffer("mask",torch.triu(torch.ones(context_length, context_length),diagonal=1))  # 预生成因果掩码，存入模型

    def forward(self, x):  # 输入形状：(2个批次, 6个词, 3维输入) (2,6,3)
        b, num_tokens, d_in = x.shape  # 获取输入形状: b=2, num_tokens=6, d_in=3
        # 一次性生成所有头的QKV  形状全部：(2,6,2)
        keys    = self.W_key(x)
        queries = self.W_query(x)
        values  = self.W_value(x)
        # 拆分多头，把最后一维拆成 (头数,每头维度) (2,6,2,1)
        keys    = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values  = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        # 转置维度，把头数放到第1位，方便并行计算  (2,6,2,1) -> (2,2,6,1)
        keys    = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values  = values.transpose(1, 2)
        # 计算所有头的注意力得分（批量矩阵乘法，transpose(最后两维) = 最后两维.T）
        attn_scores = queries @ keys.transpose(2, 3)  # (2,2,6,1) @ (2,2,1,6) = (2,2,6,6)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]  # 因果掩码，和单头一样 （缩减到当前token数量，转为布尔型）
        attn_scores.masked_fill_(mask_bool, -torch.inf)  # 按True位置遮蔽矩阵
        # 缩放+Softmax+Dropout
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        # 加权求和，得到每个头的上下文向量
        context_vec = attn_weights @ values        # (2,2,6,6) @ (2,2,6,1) = (2,2,6,1)
        context_vec = context_vec.transpose(1, 2)  # 转置回来，头数放回原来的位置 -> (2,6,2,1)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out) # 合并多头，最后两维拼在一起 -> (2,6,2)
        context_vec = self.out_proj(context_vec)   # 最终线性映射，融合所有头信息   形状不变：(2,6,2)
        return context_vec

torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape  # 2,6,3
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)  # 内部是2个独立的注意力头在并行工作，能学习到更丰富的信息

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


- 请注意，上述实现本质上是 `MultiHeadAttentionWrapper` 的重写版本，并且更加高效。  
- 生成的输出看起来略有不同，因为随机权重初始化有所不同，但两者都是完全可用的实现，可以在我们将在后续章节中实现的 GPT 类中使用。  
- 另外，值得注意的是，我们在上面的 `MultiHeadAttention` 类中添加了一个线性投影层（`self.out_proj`）。这只是一个线性变换，不改变维度。在 LLM 实现中，使用这样的投影层是标准做法，但它并非严格必要（近期的研究表明，去除该层不会影响模型的表现；请参阅本章末尾的进一步阅读部分）。  


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/26.webp" width="400px">

- 请注意，如果你对上述内容的紧凑和高效实现感兴趣，可以考虑使用PyTorch中的 [`torch.nn.MultiheadAttention`](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) 类。

- 由于上述实现初看起来可能有些复杂，我们来看一下执行 `attn_scores = queries @ keys.transpose(2, 3)` 时会发生什么：

In [43]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])
print(a.shape)
print(a @ a.transpose(2, 3))
# 每个注意力头中，输出矩阵的值表示每个 token 对其他 token 的相关性。
# 模型可以计算 token 之间的相关性，为注意力分布的生成奠定基础

torch.Size([1, 2, 3, 4])
tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


- 在这种情况下，PyTorch 中的矩阵乘法实现将处理四维输入张量，使得矩阵乘法在最后两个维度（`num_tokens`, `head_dim`）之间进行，然后对各个头进行重复计算。  

- 例如，以下方法提供了一种更紧凑的方式来分别计算每个头的矩阵乘法：  


In [45]:
first_head = a[0, 0, :, :] # 定义第一个头
first_res = first_head @ first_head.T # 第一个矩阵的自相关性(自注意力)
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


# 总结与收获

- 请参阅 [./multihead-attention.ipynb](./multihead-attention.ipynb) 代码笔记本，它是数据加载器（第2章）的简洁版本，加上我们在本章实现的多头注意力类，后续章节中训练GPT模型时将需要使用。
- 你可以在 [./exercise-solutions.ipynb](./exercise-solutions.ipynb) 中找到习题解答。